In [37]:
import arff
import pandas as pd

# загружаем данные
with open("c:/Personal/Science/Data/Binary_classification/wholesale.arff", encoding="utf-8") as f:
    dataset = arff.load(f)
df = pd.DataFrame(dataset["data"],
                  columns=[a[0] for a in dataset["attributes"]])
pd.set_option("display.max_columns", None)
df.head()

,V1,V2,V3,V4,V5,V6,V7,Channel,Region
0,3.0,12669.0,9656.0,7561.0,214.0,2674.0,1338.0,2,3
1,3.0,7057.0,9810.0,9568.0,1762.0,3293.0,1776.0,2,3
2,3.0,6353.0,8808.0,7684.0,2405.0,3516.0,7844.0,2,3
3,3.0,13265.0,1196.0,4221.0,6404.0,507.0,1788.0,1,3
4,3.0,22615.0,5410.0,7198.0,3915.0,1777.0,5185.0,2,3


In [39]:
from typing import List, Any
import numpy as np

def clean_dataset(
    df: pd.DataFrame,
    cols_to_drop: List[Any] = None,
    missing_values: List[Any] = None
) -> pd.DataFrame:
    """
    Предобработка датафрейма с учетом пользовательских меток пропусков:
    1. Заполняет пропущенные значения
       - Модой для категориальных/текстовых признаков.
       - Средним значением для числовых признаков.
    2. Удаляет признаки, у которых одно значение занимает > 90% строк.
    """
    df_cleaned = df.copy()
    n_rows = len(df_cleaned)
    
    if n_rows == 0:
        return df_cleaned

    # Задаем список меток для пропущенных значений по умолчанию, если не передан свой
    if missing_values is None:
        missing_values = ['', 'N/A', 'NA', 'null', 'nan', 'NaN', 'None', 'none', '?', '-']

    # 1. Приведение пользовательских пропусков к np.nan и их импутация
    for col in df_cleaned.columns:
        # Заменяем все кастомные метки пропусков на np.nan
        df_cleaned[col] = df_cleaned[col].replace(missing_values, np.nan)

        if df_cleaned[col].isna().sum() > 0:
            if pd.api.types.is_numeric_dtype(df_cleaned[col]):
                # а) Числовые переменные -> средним значением
                mean_val = df_cleaned[col].mean()
                df_cleaned[col] = df_cleaned[col].fillna(mean_val)
            else:
                # б) Категориальные переменные -> модой
                mode_val = df_cleaned[col].mode()
                if not mode_val.empty:
                    df_cleaned[col] = df_cleaned[col].fillna(mode_val[0])

    # 2. Проверка условий для удаления колонок
    for col in df_cleaned.columns:
        series = df_cleaned[col]
        max_value_count = series.value_counts().iloc[0]
        max_value_ratio = max_value_count / n_rows

        # Если константных значений или пропусков больше 90% -> к удалению
        if max_value_ratio > 0.9:
            cols_to_drop.append(col)

    # Удаляем выявленные колонки
    if cols_to_drop:
        df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    return df_cleaned

df = clean_dataset(df)

In [41]:
df['Channel'] = df['Channel'].map({'2' : 0, '1' : 1 })
y = df["Channel"]
X = df.drop(["Channel"], axis=1)

In [43]:
column_types = {
        "sufficient_resources_nominal_columns" : ["Region"],
        "insufficient_resources_nominal_columns": None,
        "binary": None,
        "ordinal": None,
        "inclusion": None,
        "similarity":None,
    }
ANOC_threshold = 100

In [47]:
import time
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from preprocessor import clean_dataset
from meta_encoder import MetaEncoder
from sklearn.base import clone

# models
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

#from approach import encode_with_approach

model_types = {
    "LogReg": "ATI",
    "MLP": "ATI",
    "LinearSVM": "ATI",
    "CART": "tree",
    "RF": "tree",
    "CatBoost": "tree",
    "XGBoost": "tree",
    "LightGBM": "tree"
}
target_encoders = ['Leave-one-out', 'K-fold', 'CatBoost', 'CV-GLMM', 'WOE']

nominal_cols = column_types["sufficient_resources_nominal_columns"].copy()
if column_types["insufficient_resources_nominal_columns"]:
    nominal_cols.extend(column_types["insufficient_resources_nominal_columns"])

max_categories = df[nominal_cols].nunique().max()
ANOC = df.shape[0] / max_categories
# ANOC = 101

encode_with_target: bool = False

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    "LinearSVM": LinearSVC(dual="auto", random_state=42),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():

    # initializing guide parameters
    if model_types[model_name] == 'ATI':
        if ANOC > ANOC_threshold:
            encode_with_target = False
        else:
            encode_with_target = True
    else:  # 'tree-based'
        if ANOC > ANOC_threshold:
            encode_with_target = True
        else:
            encode_with_target = False

    target_encoders_to_apply = []

    if encode_with_target:
        target_encoders_to_apply = target_encoders
    else: # it just a mark that we use target-agnostic encoders
        target_encoders_to_apply = ["OneHot"]

    for nominal_encoder in target_encoders_to_apply:

        # creating parameters for the meta-encoder
        encode_columns_as = {}
        for type, columns in column_types.items():
            if type == "binary":
                encode_columns_as["Binary"] = columns
                continue
            if type == "similarity":
                encode_columns_as["Similarity"] = columns
                continue
            if type == "ordinal":
                encode_columns_as["Ordinal"] = columns
                continue
            if type == "inclusion":
                encode_columns_as["RankHot"] = columns
                continue

        encoder_name: str
        if not encode_with_target:
            if column_types["sufficient_resources_nominal_columns"] :
                encode_columns_as["OneHot"] = column_types["sufficient_resources_nominal_columns"]
                encoder_name = "OneHot"
            if column_types["insufficient_resources_nominal_columns"] :
                encode_columns_as["Hash"] = column_types["insufficient_resources_nominal_columns"]
                if encoder_name != '':
                    encoder_name = encoder_name + "_Hash"
        else:
            encode_columns_as[nominal_encoder] = nominal_cols
            encoder_name = nominal_encoder

        encoder = MetaEncoder(encode_columns_as=encode_columns_as)

        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            X_train_encoded = encoder.fit_transform(X_train, y_train)
            X_test_encoded = encoder.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Approach",
                    "dataset": "Wholesale",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Approach, Wholesale, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_wholesale_approach.csv", index=True)
print("Done!")

Approach, Wholesale, LogReg, OneHot, 1, 1, 0.9304500292226768, 0.9473793071550753, 0.8977272727272727, 0.926829268292683, 0.890625, 0.9661016949152542, 0.04446244239807129
Approach, Wholesale, LogReg, OneHot, 1, 2, 0.9468147282291057, 0.9456197574491523, 0.9090909090909091, 0.9344262295081968, 0.9047619047619048, 0.9661016949152542, 0.037557125091552734
Approach, Wholesale, LogReg, OneHot, 1, 3, 0.9559523809523809, 0.979390542680478, 0.8863636363636364, 0.9137931034482759, 0.9464285714285714, 0.8833333333333333, 0.02864551544189453
Approach, Wholesale, LogReg, OneHot, 1, 4, 0.9607142857142856, 0.9719930304228136, 0.9090909090909091, 0.9354838709677419, 0.90625, 0.9666666666666667, 0.028882980346679688
Approach, Wholesale, LogReg, OneHot, 1, 5, 0.9773809523809524, 0.9911322036322036, 0.9204545454545454, 0.943089430894309, 0.9206349206349206, 0.9666666666666667, 0.026646137237548828
Approach, Wholesale, LogReg, OneHot, 2, 1, 0.9824663939216832, 0.9923392658308994, 0.9431818181818182, 0.9